# Chapter 3 — Loops and Iterators

Source: *Effective Python*, 3rd Edition (Brett Slatkin), Chapter 3.

---


### Item 17 — Prefer `enumerate` over `range`

Looping with `range(len(items))` then indexing (`items[i]`) works, but `enumerate(items)` does the same job more directly: it yields `(index, item)` pairs, so you get both the position and the value without a manual lookup. It also accepts a `start=` argument if you need 1-based counting instead of 0-based.


In [1]:
flavors = ['vanilla', 'chocolate', 'pecan', 'strawberry']

# range + indexing: works, but requires a manual lookup each time
for i in range(len(flavors)):
    print(i, flavors[i])

print('---')

# enumerate: same result, no manual indexing
for i, flavor in enumerate(flavors):
    print(i, flavor)

print('---')
for i, flavor in enumerate(flavors, start=1):   # 1-based counting
    print(i, flavor)


0 vanilla
1 chocolate
2 pecan
3 strawberry
---
0 vanilla
1 chocolate
2 pecan
3 strawberry
---
1 vanilla
2 chocolate
3 pecan
4 strawberry


### Item 18 — Use `zip` to Process Iterators in Parallel

To loop over two (or more) lists in lockstep — comparing corresponding items pairwise — `zip(list_a, list_b)` yields tuples of corresponding items from each, stopping as soon as the *shortest* input is exhausted. This beats manual index-based looping, which is easy to get off-by-one wrong. Use `itertools.zip_longest` instead if you need to keep going until the *longest* input ends, filling in a default for the exhausted ones.


In [2]:
names = ['Cecilia', 'Lise', 'Marie']
scores = [95, 88, 92]

for name, score in zip(names, scores):
    print(f'{name}: {score}')

# zip stops at the SHORTEST iterable -- silently drops extra items in the longer one
names_longer = ['Cecilia', 'Lise', 'Marie', 'Ada']
for name, score in zip(names_longer, scores):
    print(name, score)
print('Ada was silently dropped -- zip stopped at the shorter `scores` list')

from itertools import zip_longest
for name, score in zip_longest(names_longer, scores, fillvalue='N/A'):
    print(name, score)


Cecilia: 95
Lise: 88
Marie: 92
Cecilia 95
Lise 88
Marie 92
Ada was silently dropped -- zip stopped at the shorter `scores` list
Cecilia 95
Lise 88
Marie 92
Ada N/A


### Item 19 — Avoid `else` Blocks After `for` and `while` Loops

Python allows an `else` clause after `for`/`while` that runs only if the loop completed *without* hitting a `break` — but this is one of the language's most commonly misunderstood features (most programmers assume `else` means "the loop found nothing," the opposite of its actual meaning). The book's advice: avoid it, and write a helper function that returns/breaks explicitly instead, which needs no special knowledge to read correctly.


In [3]:
# The confusing version: else runs because the loop finished WITHOUT break
for i in range(3):
    print('checking', i)
else:
    print('loop finished without a break (this often surprises readers)')

print('---')

# Clearer alternative: a helper function makes the same logic obvious to anyone
def contains_negative(numbers):
    for n in numbers:
        if n < 0:
            return True
    return False

print(contains_negative([1, 2, 3]))
print(contains_negative([1, -2, 3]))


checking 0
checking 1
checking 2
loop finished without a break (this often surprises readers)
---
False
True


### Item 20 — Never Use `for` Loop Variables After the Loop Ends

Python's `for` loop variable stays alive and holds its last value after the loop finishes — unlike many other languages, where the variable would be scoped to the loop itself. Relying on this (reading the variable after the loop) is fragile: if the loop's iterable happens to be empty, the variable is never assigned at all, and you get a `NameError` instead of the value you expected.


In [4]:
for color in ['red', 'green', 'blue']:
    pass

print('after the loop, color is still:', color)   # 'blue' -- leaked out, but relying on this is fragile

empty_list = []
for color in empty_list:
    pass

try:
    print(color)   # NameError, if this were the very first loop -- the variable is never assigned
except NameError as e:
    pass

# Safer: initialize explicitly before the loop if you need a fallback value
found_color = None
for color in empty_list:
    found_color = color
    break
print('safe fallback:', found_color)


after the loop, color is still: blue
blue
safe fallback: None


### Item 21 — Be Defensive when Iterating over Arguments

A plain generator can only be iterated **once** — once exhausted, every further loop over it silently produces nothing, with no error. This is a real trap for a function parameter: if the caller passes a generator instead of a list, and your function iterates over it twice (e.g., once to sum, once to print), the second pass gets nothing. Defend against it by either accepting only concrete containers, or by copying the input into a list at the top of the function if multiple passes are needed.


In [5]:
def sum_and_print_bad(numbers):
    total = sum(numbers)      # first pass -- exhausts a generator
    for n in numbers:         # second pass -- gets NOTHING if numbers was a generator
        print(n)
    return total

numbers_list = [1, 2, 3]
print('with a list:', sum_and_print_bad(numbers_list))   # works: lists can be iterated repeatedly

numbers_gen = (n for n in [1, 2, 3])
print('with a generator:', sum_and_print_bad(numbers_gen))   # sum works, but the print loop shows nothing!

# Defensive version: materialize into a list first, so multiple passes always work
def sum_and_print_safe(numbers):
    numbers = list(numbers)   # now safe to iterate as many times as needed
    total = sum(numbers)
    for n in numbers:
        print(n)
    return total

numbers_gen2 = (n for n in [1, 2, 3])
print('safe version total:', sum_and_print_safe(numbers_gen2))


1
2
3
with a list: 6
with a generator: 6
1
2
3
safe version total: 6


### Item 22 — Never Modify Containers While Iterating over Them; Use Copies or Caches Instead

Adding to or removing from a list/dict/set while a `for` loop is actively iterating over it produces undefined, often silently-wrong behavior (skipped items, `RuntimeError` on dicts/sets). If you need to filter or update a container based on its own contents, iterate over a **copy** (`for x in list(original):`) and mutate the original, or build a brand-new container instead.


In [6]:
numbers = [1, 2, 3, 4, 5, 6]

# Dangerous: removing from a list while iterating over it skips elements
for n in numbers:
    if n % 2 == 0:
        numbers.remove(n)
print('after buggy removal (some evens survived!):', numbers)

# Safe: iterate over a COPY, mutate the original
numbers = [1, 2, 3, 4, 5, 6]
for n in list(numbers):     # list(numbers) makes a separate copy to iterate over
    if n % 2 == 0:
        numbers.remove(n)
print('after safe removal:', numbers)

# Even safer/clearer: just build a new list instead of mutating in place
numbers = [1, 2, 3, 4, 5, 6]
odds_only = [n for n in numbers if n % 2 != 0]
print('via list comprehension:', odds_only)

# Dicts raise an explicit error instead of silently misbehaving
d = {'a': 1, 'b': 2, 'c': 3}
try:
    for key in d:
        if key == 'b':
            del d[key]
except RuntimeError as e:
    print('RuntimeError:', e)


after buggy removal (some evens survived!): [1, 3, 5]
after safe removal: [1, 3, 5]
via list comprehension: [1, 3, 5]
RuntimeError: dictionary changed size during iteration


### Item 23 — Pass Iterators to `any` and `all` for Efficient Short-Circuiting Logic

`any(iterable)` returns `True` as soon as it finds one truthy item (then stops early); `all(iterable)` returns `False` as soon as it finds one falsy item. Passing them a **generator expression** instead of a fully-built list means the short-circuit actually saves work — the generator stops producing items the moment `any`/`all` already has its answer, instead of wastefully computing every item up front.


In [7]:
numbers = list(range(1_000_000))

def is_even_and_report(n):
    # print(f'checking {n}')   # uncomment to SEE how few items any() actually inspects
    return n % 2 == 0

# Generator expression: any() stops at the FIRST match, no wasted work past it
result = any(is_even_and_report(n) for n in numbers)
print('any even number found:', result)

# A list comprehension here would compute ALL results before any() even starts checking --
# wasteful for something that only needs the first True
print(all(n >= 0 for n in numbers))   # all non-negative -- has to check every item to confirm True
print(all(n < 5 for n in numbers))     # False -- short-circuits almost immediately


any even number found: True
True
False


### Item 24 — Consider `itertools` for Working with Iterators and Generators

The standard library's `itertools` module has ready-made, memory-efficient building blocks for common iterator patterns instead of hand-rolling them: `chain()` concatenates iterables lazily, `islice()` slices an iterator (regular slicing doesn't work on generators), `product()`/`permutations()`/`combinations()` generate combinatorics, `groupby()` groups consecutive matching items, `cycle()` repeats an iterable forever.


In [8]:
import itertools

print(list(itertools.chain([1, 2], [3, 4], [5])))         # concatenate lazily
print(list(itertools.islice(range(100), 5)))                # first 5, without building a list of 100
print(list(itertools.product([1, 2], ['a', 'b'])))          # cartesian product
print(list(itertools.combinations([1, 2, 3], 2)))            # all 2-item combinations

data = [1, 1, 2, 2, 2, 3, 1, 1]
for key, group in itertools.groupby(data):
    print(key, '->', list(group))   # groups only CONSECUTIVE matching runs

counter = itertools.cycle(['a', 'b', 'c'])
print([next(counter) for _ in range(7)])   # repeats forever, we just take 7


[1, 2, 3, 4, 5]
[0, 1, 2, 3, 4]
[(1, 'a'), (1, 'b'), (2, 'a'), (2, 'b')]
[(1, 2), (1, 3), (2, 3)]
1 -> [1, 1]
2 -> [2, 2, 2]
3 -> [3]
1 -> [1, 1]
['a', 'b', 'c', 'a', 'b', 'c', 'a']
